# Cloud workload forecasting: requirements first

**Goal:** determine whether short-history aggregate CPU forecasts improve on persistence,
while satisfying both required targets and the 110/24 chronological split. Run cells in
order. All development uses the first 110 timestamps. The last cell is the only cell
allowed to evaluate actual holdout targets, and is disabled by default.

This is an explained, executable starting implementation, not a pre-written conclusion.
Markdown distinguishes contract requirements, modeling recommendations, and limitations.

## 0. Requirements and setup — terminal, before running Python cells

Use **64-bit Python 3.12** for the tested environment. The six direct dependencies below
match your supplied versions; other installed packages are not needed for this investigation.
No LightGBM, XGBoost, seaborn, Plotly, or statsmodels dependency is required.

| Package | Version | Why it is needed |
|---|---:|---|
| NumPy | 2.4.6 | Arrays, errors, quantiles and small numerical checks |
| pandas | 3.1.0 | CSV loading, aggregation and lagged feature tables |
| SciPy | 1.17.1 | Fit the one-parameter exponential smoothing model |
| scikit-learn | 1.9.1 | Training-only scaling and Ridge autoregression |
| Matplotlib | 3.11.2 | Training-data and forecast-error plots |
| ipykernel | 7.4.0 | VS Code notebook kernel |

**Windows PowerShell**, from the project folder:

```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install --upgrade pip
.\.venv\Scripts\python.exe -m pip install --only-binary=:all: -r requirements.txt
.\.venv\Scripts\python.exe -m pip check
.\.venv\Scripts\python.exe -m ipykernel install --user --name atomity-forecasting --display-name "Python (Atomity forecasting)"
```

**macOS/Linux**, from the project folder:

```bash
python3.12 -m venv .venv
.venv/bin/python -m pip install --upgrade pip
.venv/bin/python -m pip install --only-binary=:all: -r requirements.txt
.venv/bin/python -m pip check
.venv/bin/python -m ipykernel install --user --name atomity-forecasting --display-name "Python (Atomity forecasting)"
```

If reusing your current environment, run `python -m pip check` first; the first code cell
also checks the selected kernel and imports. The separate environment is the reproducible
option. These commands do not require PowerShell activation or a change to execution policy.

Install VS Code's **Python** (`ms-python.python`) and **Jupyter** (`ms-toolsai.jupyter`)
extensions if absent. Open this `.ipynb`, click **Select Kernel**, and choose the environment
named above. A notebook kernel can differ from the terminal interpreter.

Put the bundle's contents in the challenge repository root. The data cell reuses a complete
`challenge_data/` directory or downloads only the fixed challenge archive if that directory
is empty. It never downloads upstream Azure observations.

Sources: [VS Code notebook setup](https://code.visualstudio.com/docs/datascience/jupyter-notebooks),
[scikit-learn installation](https://scikit-learn.org/stable/install.html),
[pandas 3.1 dependency changes](https://pandas.pydata.org/pandas-docs/stable/whatsnew/v3.1.0.html).


In [ ]:
import csv
import gzip
import hashlib
import importlib.metadata as metadata
import json
import math
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import tarfile
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize_scalar
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if sys.version_info < (3, 11) or sys.maxsize <= 2**32:
    raise RuntimeError('Use 64-bit Python 3.11+; Python 3.12 is the tested profile.')
EXPECTED_VERSIONS = {
    'numpy': '2.4.6', 'pandas': '3.1.0', 'scipy': '1.17.1',
    'scikit-learn': '1.9.1', 'matplotlib': '3.11.2',
    'ipykernel': '7.4.0',
}
versions = pd.DataFrame([
    {'package': name, 'installed': metadata.version(name), 'tested_pin': pin}
    for name, pin in EXPECTED_VERSIONS.items()
])
print('Kernel:', sys.executable)
print('Python:', platform.python_version(), '| platform:', platform.platform())
display(versions)
if not versions['installed'].eq(versions['tested_pin']).all():
    print('Version differences detected. Imports succeeded; use requirements.txt for the tested profile.')
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
smoke_model = make_pipeline(StandardScaler(), Ridge(alpha=1, solver='svd'))
smoke_model.fit(pd.DataFrame({'x': [0., 1., 2.]}), [0., 1., 2.])
assert np.isfinite(smoke_model.predict(pd.DataFrame({'x': [3.]}))).all()
assert minimize_scalar(lambda x: (x - .3)**2, bounds=(0, 1), method='bounded').success
plt.rcParams.update({'figure.figsize': (10, 3.5), 'axes.grid': True, 'grid.alpha': .2})
print('Dependency and numerical smoke checks passed.')


## 1. Freeze the problem and modeling decisions

**Contract:** distinct observed VM count and unweighted mean `avg_cpu`; first 110
observations for development; final 24 for one two-hour fixed-origin forecast. Required
timestamps are five minutes apart. No random row split or holdout-based model choice.

**Recommendations fixed before looking at validation scores:** compare persistence, simple
exponential smoothing, and Ridge using three expanding-window folds. Use lag 1/2/3 and a
trailing six-bin mean. Explore only three Ridge penalties. The acceptance rules below are
practical guardrails, not significance tests or assignment requirements.

**Limitations:** complete-trace selection conditions on continued observation. The short
window cannot establish daily/weekly cycles, arrivals/departures in the wider fleet,
infrastructure savings, memory utilization, or counterfactual unsatisfied CPU demand.
Bucket labels do not give exact cores or resource prices. No calendar features are used
because source timestamps are relative seconds without a documented wall-clock mapping.

The release has 134 labels but the model-development sample size is **110 time points**,
not 1.34 million independent examples. There are only three disjoint validation windows.

[Assignment](https://github.com/atomityhq/forecasting-challenge/blob/main/ASSIGNMENT.md) ·
[Dataset](https://github.com/atomityhq/forecasting-challenge/blob/main/DATASET.md) ·
[Submission](https://github.com/atomityhq/forecasting-challenge/blob/main/SUBMISSION.md) ·
[Azure semantics](https://github.com/Azure/AzurePublicDataset/blob/master/AzurePublicDatasetV2.md)


In [ ]:
PROJECT_ROOT = Path(os.environ.get('CWFC_ROOT', Path.cwd())).resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = Path(os.environ.get('CWFC_DATA', PROJECT_ROOT / 'challenge_data')).resolve()
OUTPUT_DIR = Path(os.environ.get('CWFC_OUTPUT', PROJECT_ROOT / 'reports/generated')).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SIZE, HORIZON, CADENCE = 110, 24, 300
TARGETS = ['active_vm_count', 'aggregate_cpu_utilization']
ORIGINS = (38, 62, 86)
CANDIDATES = ('persistence', 'ses', 'ridge_1', 'ridge_10', 'ridge_100')
FEATURE_NAMES = ['lag_1', 'lag_2', 'lag_3', 'mean_6']
ZERO_ATOL = 1e-12
MIN_GAIN = .05
MAX_PEAK_DEGRADATION = .05
PEAK_QUANTILE = .90
CONFIG = {
    'train_size': TRAIN_SIZE, 'horizon': HORIZON, 'cadence_seconds': CADENCE,
    'origins': list(ORIGINS), 'candidates': list(CANDIDATES),
    'features': FEATURE_NAMES, 'peak_quantile': PEAK_QUANTILE,
    'minimum_mae_gain': MIN_GAIN, 'minimum_winning_windows': 2,
    'maximum_peak_mae_degradation': MAX_PEAK_DEGRADATION,
    'sensitivity_rule': 'retain persistence if selected model loses MAE advantage under shifted origins, delay, or leave-one-window-out',
    'bounds': 'CPU [0,100]; count >=0; applied at every recursive step',
}
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Outputs:', OUTPUT_DIR)
display(pd.DataFrame([
    {'training_rows': o, 'validation_first_row': o + 1, 'validation_last_row': o + HORIZON}
    for o in ORIGINS
]))


## 2. Acquire only `data-v1` and verify its bytes

The source archive hash is pinned to the challenge release. Existing files must have a
valid `SHA256SUMS`; an incomplete directory is rejected instead of silently overwritten.
The downloader extracts only the four named regular files, avoiding unrestricted archive
extraction. A SHA-256 match establishes byte identity, not population representativeness.

Read the **actual bundled `DATA_CARD.md`**, rather than assuming its template is identical.
The documented complete-trace eligibility rule explains why observed population count is
expected to be flat. Its upstream sampling mechanism cannot be independently reconstructed
from the selected records alone.


In [ ]:
RELEASE_URL = 'https://github.com/atomityhq/forecasting-challenge/releases/download/data-v1/data-v1.tar.gz'
ARCHIVE_SHA256 = 'd04d402ac1719c1f5b869e3699fc3d57f6f8273751f1ef19bedb29397b39caeb'
RELEASE_FILES = {'vm_cpu_readings.csv.gz', 'vm_metadata.csv.gz', 'DATA_CARD.md', 'SHA256SUMS'}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def ensure_challenge_release(directory):
    directory.mkdir(parents=True, exist_ok=True)
    present = {name for name in RELEASE_FILES if (directory / name).is_file()}
    if present and present != RELEASE_FILES:
        raise ValueError(f'Incomplete challenge_data directory; missing {sorted(RELEASE_FILES - present)}. Complete extraction first.')
    if not present:
        archive = directory.parent / 'data-v1.tar.gz'
        if not archive.exists():
            print('Downloading the fixed challenge release (about 117 MB).')
            temporary = archive.with_suffix(archive.suffix + '.part')
            with urllib.request.urlopen(RELEASE_URL, timeout=180) as response, temporary.open('wb') as out:
                shutil.copyfileobj(response, out)
            temporary.replace(archive)
        if sha256_file(archive) != ARCHIVE_SHA256:
            raise ValueError('Archive checksum mismatch. Do not use these bytes; remove the bad archive and download again.')
        with tarfile.open(archive, 'r:gz') as bundle:
            for name in sorted(RELEASE_FILES):
                member = bundle.getmember(name)
                if not member.isfile():
                    raise ValueError(f'Expected a regular file: {name}')
                with bundle.extractfile(member) as source, (directory / name).open('wb') as out:
                    shutil.copyfileobj(source, out)
    checksums = {}
    for line in (directory / 'SHA256SUMS').read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        expected, name = line.split(maxsplit=1)
        name = name.lstrip('*')
        if name not in RELEASE_FILES - {'SHA256SUMS'} or len(expected) != 64:
            raise ValueError('Unexpected entry in SHA256SUMS.')
        actual = sha256_file(directory / name)
        if actual != expected:
            raise ValueError(f'Content checksum mismatch: {name}')
        checksums[name] = actual
    if set(checksums) != RELEASE_FILES - {'SHA256SUMS'}:
        raise ValueError('SHA256SUMS does not cover the three content files.')
    return checksums

data_hashes = ensure_challenge_release(DATA_DIR)
display(pd.DataFrame([{'file': name, 'sha256': value} for name, value in data_hashes.items()]))
print((DATA_DIR / 'DATA_CARD.md').read_text(encoding='utf-8'))


## 3. Preprocessing and exact target construction

**Validity interpretation:** a usable observation has a VM ID, integer timestamp and finite
`avg_cpu` in `[0,100]`. Missing/invalid averages are excluded from both targets. Exact duplicate
rows are removed; conflicting rows for the same VM/timestamp raise an error. Contradictory
finite min/average/max readings also raise an error. Missing min/max does not change a valid
average's weight, but is audited. No target is filled, interpolated, standardized or smoothed.

Count distinct VMs after that rule. Compute the ordinary mean of one average per VM. A VM
with 2 cores and one in `>24` get equal weight. We do not convert bucket labels to exact cores.
A completely missing timestamp or a timestamp without a usable average is a failure: CPU
utilization there is unknown, not zero.

Aggregate the **full series first**, as required, then return only the training targets and
a timestamp-only future template. The full target table stays local to the loader and is
discarded. Later exploratory cells receive only training CPU readings. Structural checks
across the release inspect keys/population coverage; holdout CPU levels are not displayed.


In [ ]:
CPU_COLUMNS = ['timestamp', 'vm_id', 'min_cpu', 'max_cpu', 'avg_cpu']
META_COLUMNS = ['vm_id', 'subscription_id', 'deployment_id', 'timestamp_vm_created',
                'timestamp_vm_deleted', 'vm_category', 'virtual_core_bucket', 'memory_gb_bucket']

def clean_cpu(frame):
    if list(frame.columns) != CPU_COLUMNS:
        raise ValueError(f'CPU schema must be {CPU_COLUMNS}.')
    frame = frame.copy()
    if frame['timestamp'].isna().any() or frame['vm_id'].isna().any() or frame['vm_id'].astype(str).str.strip().eq('').any():
        raise ValueError('Missing observation key.')
    timestamps = pd.to_numeric(frame['timestamp'], errors='raise').to_numpy(dtype=float)
    if not np.isfinite(timestamps).all() or (timestamps < 0).any() or (timestamps != np.floor(timestamps)).any():
        raise ValueError('Timestamps must be nonnegative integer relative seconds.')
    frame['timestamp'] = timestamps.astype('int64')
    for column in ['min_cpu', 'max_cpu', 'avg_cpu']:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
    original_rows = len(frame)
    frame = frame.drop_duplicates()
    if frame.duplicated(['timestamp', 'vm_id']).any():
        raise ValueError('Conflicting duplicate VM/timestamp pairs.')
    valid = np.isfinite(frame['avg_cpu']) & frame['avg_cpu'].between(0, 100)
    missing_minmax = ~np.isfinite(frame[['min_cpu', 'max_cpu']].to_numpy(dtype=float)).all(axis=1)
    finite_bounds = ~missing_minmax
    contradictions = valid & finite_bounds & (
        (frame['min_cpu'] < 0) | (frame['max_cpu'] > 100) |
        (frame['min_cpu'] > frame['avg_cpu']) | (frame['avg_cpu'] > frame['max_cpu'])
    )
    if contradictions.any():
        raise ValueError('Contradictory min/average/max CPU readings.')
    audit = {'source_rows': original_rows, 'exact_duplicates_removed': original_rows - len(frame),
             'invalid_averages_excluded': int((~valid).sum()),
             'rows_with_missing_minmax': int(missing_minmax.sum())}
    return frame.loc[valid].copy(), audit

def aggregate_targets(frame, timestamps=None):
    clean, audit = clean_cpu(frame)
    grid = np.sort(frame['timestamp'].unique()) if timestamps is None else np.asarray(timestamps)
    if len(grid) < 2 or not np.all(np.diff(grid) == CADENCE):
        raise ValueError('Expected an uninterrupted 300-second timestamp grid.')
    targets = clean.groupby('timestamp', observed=True).agg(
        active_vm_count=('vm_id', 'nunique'), aggregate_cpu_utilization=('avg_cpu', 'mean')
    ).reindex(grid)
    if targets.isna().any().any():
        raise ValueError('At least one timestamp has no usable observations. No target imputation is applied.')
    targets['active_vm_count'] = targets['active_vm_count'].astype('int64')
    targets = targets.rename_axis('timestamp').reset_index()
    return targets, clean, audit

def development_partition(targets, train_size=TRAIN_SIZE, horizon=HORIZON):
    if len(targets) != train_size + horizon:
        raise ValueError(f'Expected {train_size + horizon} timestamps, found {len(targets)}.')
    return targets.iloc[:train_size].copy(), targets.iloc[train_size:][['timestamp']].copy()

def load_development(directory):
    raw = pd.read_csv(directory / 'vm_cpu_readings.csv.gz', dtype={'vm_id': 'category'})
    meta = pd.read_csv(directory / 'vm_metadata.csv.gz', usecols=META_COLUMNS, dtype='string')
    if meta['vm_id'].isna().any() or meta['vm_id'].duplicated().any():
        raise ValueError('Metadata must have one row per nonmissing VM ID.')
    if set(raw['vm_id'].dropna()) != set(meta['vm_id']):
        raise ValueError('CPU and metadata VM populations differ.')
    targets, clean, quality = aggregate_targets(raw)
    train, future = development_partition(targets)
    per_vm = raw.drop_duplicates(['timestamp', 'vm_id']).groupby('vm_id', observed=True)['timestamp'].nunique()
    structure = {
        'cpu_rows': len(raw), 'cpu_vms': int(raw['vm_id'].nunique()),
        'metadata_rows': len(meta), 'timestamps': len(targets),
        'first_timestamp': int(targets['timestamp'].iloc[0]),
        'last_timestamp': int(targets['timestamp'].iloc[-1]),
        'per_vm_timestamp_count_min': int(per_vm.min()),
        'per_vm_timestamp_count_max': int(per_vm.max()),
        'all_vms_cover_full_grid': bool(per_vm.eq(len(targets)).all()),
        **quality,
    }
    training_cpu = clean.loc[clean['timestamp'] <= train['timestamp'].iloc[-1]].copy()
    return train, future, training_cpu, meta, structure

train, future_timestamps, training_cpu, vm_metadata, data_audit = load_development(DATA_DIR)
display(pd.Series(data_audit, name='structural / quality audit').to_frame())
display(train.head())
print('Development rows:', len(train), '| future timestamps only:', len(future_timestamps))
print('Training end:', train['timestamp'].iloc[-1], '| forecast range:',
      future_timestamps['timestamp'].iloc[0], 'to', future_timestamps['timestamp'].iloc[-1])


## 4. Verify aggregation independently and by hand

An independent streaming calculation reads training timestamps from the raw compressed CSV.
It uses distinct VM dictionaries and `math.fsum`, without calling the pandas aggregator.
Agreement is exact for counts and within `1e-10` percentage points for CPU means.

Hand fixtures check two averages (10, 30 → 20), duplicate handling, zero CPU, a missing
average, conflicting duplicates and a missing timestamp. These test semantics rather than
merely comparing a function with itself.


In [ ]:
def independent_training_aggregation(path, wanted_timestamps):
    buckets = {int(t): {} for t in wanted_timestamps}
    with gzip.open(path, 'rt', newline='') as stream:
        for row in csv.DictReader(stream):
            t = int(row['timestamp'])
            if t not in buckets:
                continue
            value = float(row['avg_cpu'])
            if not math.isfinite(value) or not 0 <= value <= 100:
                continue
            vm_id = sys.intern(row['vm_id'])
            previous = buckets[t].get(vm_id)
            if previous is not None and previous != value:
                raise ValueError('Conflicting duplicate in independent aggregation.')
            buckets[t][vm_id] = value
    return pd.DataFrame([
        {'timestamp': t, 'active_vm_count': len(values),
         'aggregate_cpu_utilization': math.fsum(values.values()) / len(values)}
        for t, values in sorted(buckets.items())
    ])

def expect_value_error(function):
    try:
        function()
    except ValueError:
        return
    raise AssertionError('Expected ValueError.')

fixture = pd.DataFrame([
    [9000, 'a', 10., 10., 10.], [9000, 'b', 30., 30., 30.],
    [9300, 'a', 0., 0., 0.], [9300, 'b', 20., 20., 20.],
], columns=CPU_COLUMNS)
expected, _, _ = aggregate_targets(fixture)
np.testing.assert_allclose(expected['aggregate_cpu_utilization'], [20., 10.])
assert expected['active_vm_count'].tolist() == [2, 2]
duplicated, _, _ = aggregate_targets(pd.concat([fixture, fixture.iloc[[0]]], ignore_index=True))
pd.testing.assert_frame_equal(expected, duplicated)
missing = fixture.copy()
missing.loc[1, 'avg_cpu'] = np.nan
missing_targets, _, _ = aggregate_targets(missing)
assert missing_targets.loc[0, 'active_vm_count'] == 1
assert missing_targets.loc[0, 'aggregate_cpu_utilization'] == 10
conflict = fixture.iloc[[0]].copy()
conflict[['min_cpu', 'max_cpu', 'avg_cpu']] = 50.
expect_value_error(lambda: aggregate_targets(pd.concat([fixture, conflict], ignore_index=True)))
gap = fixture.copy()
gap.loc[gap['timestamp'].eq(9300), 'timestamp'] = 9600
expect_value_error(lambda: aggregate_targets(gap))
independent = independent_training_aggregation(DATA_DIR / 'vm_cpu_readings.csv.gz', train['timestamp'])
np.testing.assert_array_equal(independent['active_vm_count'], train['active_vm_count'])
np.testing.assert_allclose(independent['aggregate_cpu_utilization'], train['aggregate_cpu_utilization'], rtol=0, atol=1e-10)
print('Hand-calculated fixtures and independent raw-data aggregation passed.')


## 5. Understand the training workload

Inspect distribution, change size, peaks and VM-level tails. The cohort mean can be steady
while individual VMs run hot. `max_cpu` describes each VM's interval maximum; maxima across
VMs need not occur simultaneously. The plot of hot VMs is descriptive and is not a third
forecasting target.

Population activity changes below mean changes in observation presence. Complete-trace
selection makes them a poor proxy for general fleet arrival/departure dynamics.


In [ ]:
training_summary = train[TARGETS].describe().T
training_summary['mean_absolute_step_change'] = train[TARGETS].diff().abs().mean()
display(training_summary)
display(training_cpu[['min_cpu', 'avg_cpu', 'max_cpu']].describe(percentiles=[.5, .9, .95, .99]).T)

hours = (train['timestamp'] - train['timestamp'].iloc[0]) / 3600
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(hours, train['aggregate_cpu_utilization'])
axes[0].set(xlabel='Hours since first supplied timestamp', ylabel='Mean VM CPU (%)', title='Training cohort mean')
axes[1].hist(training_cpu['avg_cpu'], bins=40, color='tab:blue')
axes[1].set(xlabel='VM interval average CPU (%)', ylabel='VM/readings', title='Training utilization distribution')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_workload.png', dpi=140)
plt.show()

presence = training_cpu.groupby('timestamp', observed=True)['vm_id'].nunique()
hot_share = training_cpu.assign(hot=training_cpu['avg_cpu'] >= 80).groupby('timestamp')['hot'].mean() * 100
print('Observed count range:', int(presence.min()), 'to', int(presence.max()))
print('Observed count changes:', int(presence.diff().fillna(0).ne(0).sum()))
print(f"Training share of VM readings at >=80%: {100 * training_cpu['avg_cpu'].ge(80).mean():.3f}%")
display(pd.DataFrame({'timestamp': hot_share.index, 'vm_share_at_or_above_80_percent': hot_share.values}).head())
print('80% is a descriptive marker, not a validated SLA or scaling threshold.')


## 6. Segments and retrospective metadata

Compare only category and core bucket, keeping the analysis small. Labels describe the
curated sample; source-category timing is not established, so category is not a predictor.
Metadata CPU summary columns were deliberately not loaded.

Creation/deletion timestamps are retrospective fields. A zero creation timestamp at the
trace boundary does not prove that a VM was newly provisioned then. Deletion after the
training origin cannot be used for churn or age/lifetime features at that origin. Report
the recorded span without calling it an uncensored physical VM lifetime.


In [ ]:
joined_training = training_cpu.merge(vm_metadata, on='vm_id', how='left', validate='many_to_one')
segment_tables = []
for field in ['vm_category', 'virtual_core_bucket']:
    segments = joined_training.groupby(field, observed=True).agg(
        vms=('vm_id', 'nunique'), readings=('avg_cpu', 'size'), mean_cpu=('avg_cpu', 'mean'),
        p95_vm_interval_cpu=('avg_cpu', lambda values: values.quantile(.95)),
    ).reset_index()
    segments.insert(0, 'segment_field', field)
    segments = segments.rename(columns={field: 'segment'})
    segment_tables.append(segments)
    display(segments)
segment_summary = pd.concat(segment_tables, ignore_index=True)
segment_summary.to_csv(OUTPUT_DIR / 'training_segments.csv', index=False)
recorded_creation = pd.to_numeric(vm_metadata['timestamp_vm_created'], errors='raise')
recorded_deletion = pd.to_numeric(vm_metadata['timestamp_vm_deleted'], errors='raise')
display(pd.DataFrame({
    'field': ['recorded_creation', 'recorded_deletion', 'recorded_span_seconds'],
    'min': [recorded_creation.min(), recorded_deletion.min(), (recorded_deletion - recorded_creation).min()],
    'max': [recorded_creation.max(), recorded_deletion.max(), (recorded_deletion - recorded_creation).max()],
}))
del joined_training


## 7. Diagnose persistence without inventing seasonality

Compute a short-lag sample autocorrelation for levels and first differences, using NumPy.
The normalization is the usual biased sample ACF: lagged centered cross-products divided
by the lag-zero sum of squares. Constant series have undefined nonzero-lag ACF.

Strong level ACF can simply reflect a slowly moving level. Dependence in changes is a more
useful hint of information beyond persistence. These are descriptive diagnostics on 110
points; there is no justified 288-bin daily seasonal lag in this sample. Features remain
the predeclared set rather than being searched across many diagnostic-driven alternatives.


In [ ]:
def short_acf(values, max_lag=12):
    values = np.asarray(values, dtype=float)
    centered = values - values.mean()
    denominator = centered @ centered
    return np.array([
        1.0 if lag == 0 else (float(centered[lag:] @ centered[:-lag] / denominator)
                              if denominator > ZERO_ATOL else np.nan)
        for lag in range(min(max_lag, len(values) - 1) + 1)
    ])

level_acf = short_acf(train['aggregate_cpu_utilization'])
change_acf = short_acf(np.diff(train['aggregate_cpu_utilization']))
acf_table = pd.DataFrame({'lag_bins': range(len(level_acf)), 'level_acf': level_acf, 'change_acf': change_acf})
display(acf_table)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, values, title in zip(axes, [level_acf, change_acf], ['CPU levels', 'CPU first differences']):
    ax.stem(range(len(values)), values)
    ax.set(xlabel='Lag (five-minute bins)', ylabel='Sample ACF', title=title)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'training_acf.png', dpi=140)
plt.show()


## 8. Availability audit and feature construction

| Input | Decision |
|---|---|
| Relative timestamp | Used for ordering/output; no invented hour/day/week |
| Last completed CPU/count observations | Available after the interval completes; one-bin delay tested later |
| Lag 1, 2, 3 | Shifted by 1/2/3 before matching each training label |
| Six-bin mean | `shift(1).rolling(6)`: excludes the row being predicted |
| Scaler | Fitted only on each origin's supervised training rows |
| Metadata summary CPU, deletion and full lifetime | Excluded |
| Category/resource bucket labels | Descriptive only; no metadata forecasting features |
| Future actual CPU/count | Never supplied to fitting or recursive feature updates |

The first six supervised rows are discarded because they lack complete history. They are
not filled using later observations. The small feature set represents immediate behavior
and a 30-minute local level. Correlated features motivate Ridge regularization. Scaling
gives the penalty comparable units across features and is learned within each fold.


In [ ]:
def supervised_features(values):
    series = pd.Series(np.asarray(values, dtype=float))
    features = pd.DataFrame({
        'lag_1': series.shift(1), 'lag_2': series.shift(2), 'lag_3': series.shift(3),
        'mean_6': series.shift(1).rolling(6, min_periods=6).mean(),
    })
    return features.iloc[6:].copy(), series.iloc[6:].copy()

feature_example, label_example = supervised_features(train['aggregate_cpu_utilization'])
display(feature_example.head().assign(label=label_example.head()))
fixture_features, fixture_labels = supervised_features(np.arange(10, dtype=float))
np.testing.assert_allclose(fixture_features.iloc[0], [5, 4, 3, 2.5])
assert fixture_labels.iloc[0] == 6
assert list(feature_example.columns) == FEATURE_NAMES
print('Feature alignment fixture passed. No current/future label is an input.')


## 9. Fit the three forecast families

**Persistence:** repeat the last observed value. **Simple exponential smoothing (SES):**
initialize the level at the first observation, fit one smoothing coefficient by training
one-step squared error, and repeat the final level for all future steps. Compare the
bounded optimizer's result with alpha=0 and alpha=1 endpoints. This deliberately omits
trend and seasonality. SciPy already exists in your environment, so statsmodels is not
needed just for this small level model.

**Ridge autoregression:** fit a standardized one-step regression, then recursively forecast
24 steps. Recompute each later rolling mean from actual history plus earlier predictions.
Never substitute actual validation readings during the forecast path.

Bounds are fixed physical projections: CPU `[0,100]`, count `>=0`. CPU bounds are applied at
every recursive step and clipping is counted. A constant prefix, or fewer than 12 readings,
uses persistence. This makes tiny fixtures and degenerate count targets explicit rather
than allowing unstable or meaningless fits. Count predictions need not be rounded by the contract.


In [ ]:
def forecast_values(values, candidate, target, horizon=HORIZON):
    y = np.asarray(values, dtype=float)
    if y.ndim != 1 or not len(y) or not np.isfinite(y).all() or horizon <= 0:
        raise ValueError('A nonempty finite history and positive horizon are required.')
    if candidate not in CANDIDATES or target not in TARGETS:
        raise ValueError('Unknown model or target.')
    info = {'candidate': candidate, 'effective_model': candidate, 'clipped_steps': 0}
    upper = 100.0 if target == 'aggregate_cpu_utilization' else np.inf
    if candidate == 'persistence' or len(y) < 12 or np.ptp(y) <= ZERO_ATOL:
        info['effective_model'] = 'persistence'
        return np.full(horizon, np.clip(y[-1], 0, upper)), info
    if candidate == 'ses':
        def smooth(alpha):
            level, squared_errors = y[0], []
            for observed in y[1:]:
                squared_errors.append((observed - level)**2)
                level = alpha * observed + (1 - alpha) * level
            return float(np.mean(squared_errors)), float(level)
        result = minimize_scalar(lambda alpha: smooth(alpha)[0], bounds=(0, 1), method='bounded')
        if not result.success:
            raise RuntimeError(f'SES optimization failed: {result.message}')
        alpha = min([0.0, 1.0, float(result.x)], key=lambda value: smooth(value)[0])
        info['smoothing_alpha'] = alpha
        return np.full(horizon, np.clip(smooth(alpha)[1], 0, upper)), info
    penalty = float(candidate.split('_')[1])
    features, labels = supervised_features(y)
    model = make_pipeline(StandardScaler(), Ridge(alpha=penalty, solver='svd'))
    model.fit(features, labels)
    info.update({'ridge_alpha': penalty, 'scaler_mean': model[0].mean_.tolist(),
                 'scaler_scale': model[0].scale_.tolist(),
                 'coefficients': model[1].coef_.tolist(), 'intercept': float(model[1].intercept_)})
    history, predictions = y.tolist(), []
    for _ in range(horizon):
        next_row = pd.DataFrame([[history[-1], history[-2], history[-3], np.mean(history[-6:])]], columns=FEATURE_NAMES)
        raw_prediction = float(model.predict(next_row)[0])
        prediction = float(np.clip(raw_prediction, 0, upper))
        if not np.isfinite(prediction):
            raise ValueError('Nonfinite forecast.')
        info['clipped_steps'] += int(prediction != raw_prediction)
        predictions.append(prediction)
        history.append(prediction)
    return np.asarray(predictions), info

for name in CANDIDATES:
    constant_forecast, _ = forecast_values(np.full(20, 7.), name, 'active_vm_count')
    np.testing.assert_array_equal(constant_forecast, np.full(HORIZON, 7.))
print('Constant-target fallback works for every candidate, without a hard-coded VM count.')


## 10. Expanding-window backtests

All models use identical training prefixes and 24-step validation blocks. The optional
`delay=1` sensitivity removes the most recent bin from available information. It forecasts
25 steps and scores steps 2–25, preserving the original requested future timestamps.
Using 24 steps and scoring them without that offset would be a timing error.

Peak thresholds come only from the origin's available training prefix. Validation actuals
classify peak periods for scoring; they never become predictor inputs. Validation scores
are reused for model selection, so they are development estimates, not an unbiased final test.


In [ ]:
def backtest(frame, origins=ORIGINS, delay=0, candidates=CANDIDATES, peak_quantile=PEAK_QUANTILE):
    rows, fits = [], []
    if delay < 0:
        raise ValueError('Delay must be nonnegative.')
    for origin in origins:
        if origin - delay < 1 or origin + HORIZON > len(frame):
            raise ValueError('Backtest origin exceeds the supplied development frame.')
        prefix = frame.iloc[:origin - delay]
        actual = frame.iloc[origin:origin + HORIZON]
        for target in TARGETS:
            threshold = float(prefix[target].quantile(peak_quantile))
            baseline, _ = forecast_values(prefix[target], 'persistence', target, HORIZON + delay)
            for candidate in candidates:
                forecast, info = forecast_values(prefix[target], candidate, target, HORIZON + delay)
                forecast = forecast[delay:]
                fits.append({'origin': origin, 'target': target, **info})
                for step, (timestamp, observed, predicted, base) in enumerate(zip(
                    actual['timestamp'], actual[target], forecast, baseline[delay:]
                ), start=1):
                    rows.append({'origin': origin, 'target': target, 'candidate': candidate,
                                 'horizon': step, 'timestamp': int(timestamp), 'actual': float(observed),
                                 'prediction': float(predicted), 'baseline': float(base),
                                 'peak_threshold': threshold,
                                 'is_peak': bool(target == 'aggregate_cpu_utilization' and observed >= threshold)})
    return pd.DataFrame(rows), fits

validation_predictions, validation_fits = backtest(train)
validation_predictions.to_csv(OUTPUT_DIR / 'validation_predictions.csv', index=False)
print('All validation forecast paths contain 24 steps; actuals enter only the scoring records.')


## 11. Metrics, zero-baseline handling and model selection

CPU MAE/RMSE are **percentage points**, not relative percentages. Count errors are in VMs.
Report pooled errors, window errors, peak errors and horizon errors. Avoid MAPE/sMAPE here.

Relative MAE = model MAE / persistence MAE; skill = 1 − relative MAE. When baseline error
is zero (within numerical tolerance), both are **N/A**, with an explicit exact-tie/worse
status and absolute errors. Do not add a denominator epsilon or interpret an exact tie as
improvement. Missing peak observations likewise produce N/A and an observation count.

**Predeclared recommendation:** a candidate needs at least 5% pooled MAE improvement,
wins in two of three windows, no pooled RMSE degradation and no more than 5% peak-MAE
degradation. At least three peak points are required to apply that guard; this minimum
is a small-sample heuristic, not adequate evidence of statistical significance. Among
qualifying models within 1% of the best MAE, prefer SES, then more strongly regularized Ridge.
A zero-error baseline is retained. We apply sensitivity checks before the final choice.


In [ ]:
def error_metrics(actual, prediction, baseline):
    actual, prediction, baseline = [np.asarray(values, dtype=float) for values in (actual, prediction, baseline)]
    if actual.shape != prediction.shape or actual.shape != baseline.shape or not actual.size:
        raise ValueError('Metrics require matching nonempty arrays.')
    if not all(np.isfinite(values).all() for values in (actual, prediction, baseline)):
        raise ValueError('Metrics require finite arrays.')
    error, base_error = prediction - actual, baseline - actual
    mae = float(np.mean(np.abs(error)))
    base_mae = float(np.mean(np.abs(base_error)))
    zero = base_mae <= ZERO_ATOL
    return {'mae': mae, 'rmse': float(np.sqrt(np.mean(error**2))), 'baseline_mae': base_mae,
            'relative_mae': np.nan if zero else mae / base_mae,
            'skill': np.nan if zero else 1 - mae / base_mae,
            'baseline_status': ('zero-error exact tie' if mae <= ZERO_ATOL else 'zero-error baseline; model worse') if zero else 'nonzero-error baseline'}

def summarize(predictions, group_columns=('target', 'candidate')):
    rows = []
    for key, group in predictions.groupby(list(group_columns), sort=False):
        key = key if isinstance(key, tuple) else (key,)
        result = dict(zip(group_columns, key))
        result.update(error_metrics(group['actual'], group['prediction'], group['baseline']))
        peaks = group.loc[group['is_peak']]
        result.update({'n': len(group), 'peak_n': len(peaks),
                       'peak_mae': float(np.mean(np.abs(peaks['prediction'] - peaks['actual']))) if len(peaks) else np.nan,
                       'peak_rmse': float(np.sqrt(np.mean((peaks['prediction'] - peaks['actual'])**2))) if len(peaks) else np.nan,
                       'peak_underprediction': float(np.mean(np.maximum(peaks['actual'] - peaks['prediction'], 0))) if len(peaks) else np.nan})
        rows.append(result)
    return pd.DataFrame(rows)

def choose_models(predictions):
    summary = summarize(predictions)
    windows = summarize(predictions, ('target', 'candidate', 'origin'))
    decisions = {}
    preference = {'ses': 0, 'ridge_100': 1, 'ridge_10': 2, 'ridge_1': 3}
    for target in TARGETS:
        scores = summary.loc[summary['target'].eq(target)].set_index('candidate')
        baseline = scores.loc['persistence']
        qualified = []
        if baseline['mae'] > ZERO_ATOL:
            for candidate in CANDIDATES[1:]:
                score = scores.loc[candidate]
                window_scores = windows.loc[windows['target'].eq(target) & windows['candidate'].eq(candidate)]
                wins = int((window_scores['mae'] < window_scores['baseline_mae'] - ZERO_ATOL).sum())
                peak_safe = (score['peak_n'] >= 3 and score['peak_mae'] <=
                             baseline['peak_mae'] * (1 + MAX_PEAK_DEGRADATION) + ZERO_ATOL)
                if (score['mae'] <= baseline['mae'] * (1 - MIN_GAIN) and wins >= 2
                        and score['rmse'] <= baseline['rmse'] + ZERO_ATOL and peak_safe):
                    qualified.append(candidate)
        if qualified:
            best_mae = min(scores.loc[name, 'mae'] for name in qualified)
            close = [name for name in qualified if scores.loc[name, 'mae'] <= best_mae * 1.01 + ZERO_ATOL]
            chosen = min(close, key=lambda name: preference[name])
            reason = 'Passed the predeclared MAE/window/RMSE/peak gate; sensitivity confirmation follows.'
        else:
            chosen = 'persistence'
            reason = 'Exact baseline retained.' if baseline['mae'] <= ZERO_ATOL else 'Insufficient evidence to clear every predeclared improvement guard.'
        decisions[target] = {'candidate': chosen, 'reason': reason}
    return decisions, summary, windows

assert error_metrics([1, 2, 3], [1, 3, 1], [1, 1, 1])['mae'] == 1
assert np.isnan(error_metrics([7, 7], [7, 7], [7, 7])['relative_mae'])
assert 'worse' in error_metrics([7, 7], [8, 8], [7, 7])['baseline_status']
initial_decisions, model_summary, window_summary = choose_models(validation_predictions)
display(model_summary.round(6))
display(window_summary[['target', 'candidate', 'origin', 'mae', 'rmse', 'skill']].round(6))
print(json.dumps(initial_decisions, indent=2))
model_summary.to_csv(OUTPUT_DIR / 'validation_model_summary.csv', index=False)
window_summary.to_csv(OUTPUT_DIR / 'validation_window_summary.csv', index=False)


## 12. Horizon-level and peak-period failure analysis

Each single horizon has only three observations per model. Interpret the shape alongside
broader 1–6, 7–12 and 13–24 step bands. Display the largest validation errors to inspect
abrupt changes, delayed response and recursive drift. These are failure descriptions;
telemetry alone does not establish the causal reason for a change.


In [ ]:
horizon_summary = summarize(validation_predictions, ('target', 'candidate', 'horizon'))
banded = validation_predictions.copy()
banded['horizon_band'] = pd.cut(banded['horizon'], bins=[0, 6, 12, 24], labels=['1-6', '7-12', '13-24'])
band_summary = summarize(banded, ('target', 'candidate', 'horizon_band'))
display(band_summary.loc[band_summary['target'].eq('aggregate_cpu_utilization')].round(6))
horizon_summary.to_csv(OUTPUT_DIR / 'validation_horizon_summary.csv', index=False)
band_summary.to_csv(OUTPUT_DIR / 'validation_horizon_bands.csv', index=False)

cpu_errors = validation_predictions.loc[validation_predictions['target'].eq('aggregate_cpu_utilization')].copy()
cpu_errors['absolute_error'] = (cpu_errors['prediction'] - cpu_errors['actual']).abs()
display(cpu_errors.sort_values('absolute_error', ascending=False).head(10)[
    ['candidate', 'origin', 'horizon', 'timestamp', 'actual', 'prediction', 'absolute_error', 'is_peak']])
fig, ax = plt.subplots()
for candidate, group in horizon_summary.loc[horizon_summary['target'].eq('aggregate_cpu_utilization')].groupby('candidate'):
    ax.plot(group['horizon'], group['mae'], label=candidate)
ax.set(xlabel='Forecast horizon (five-minute steps)', ylabel='MAE (percentage points)', title='Development errors by horizon — only 3 examples per step')
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'horizon_errors.png', dpi=140)
plt.show()


## 13. Small, fixed sensitivity checks

Reuse the original candidates; do not search for replacements after a failed sensitivity.
Shift origins six bins earlier, impose a one-bin observation delay, and remove each original
validation window in turn. Overlapping alternatives are not independent replications.
Report peak errors again at training-prefix 85th/95th percentile thresholds without changing
any predictions. If the initially selected model loses its MAE advantage in any origin,
delay or leave-one-window-out summary, fall back to persistence and report fragility.


In [ ]:
sensitivity_rows = []
shifted_predictions, _ = backtest(train, origins=tuple(o - 6 for o in ORIGINS))
delayed_predictions, _ = backtest(train, delay=1)
alternatives = [('origins_minus_6', shifted_predictions), ('one_bin_delay', delayed_predictions)]
alternatives += [(f'leave_out_origin_{origin}', validation_predictions.loc[validation_predictions['origin'].ne(origin)]) for origin in ORIGINS]
final_decisions = {target: dict(decision) for target, decision in initial_decisions.items()}
for name, predictions in alternatives:
    for target in TARGETS:
        candidate = initial_decisions[target]['candidate']
        subset = predictions.loc[predictions['target'].eq(target) & predictions['candidate'].eq(candidate)]
        metrics = error_metrics(subset['actual'], subset['prediction'], subset['baseline'])
        sensitivity_rows.append({'check': name, 'target': target, 'candidate': candidate, **metrics})
        if candidate != 'persistence' and metrics['mae'] >= metrics['baseline_mae'] - ZERO_ATOL:
            final_decisions[target] = {'candidate': 'persistence', 'reason': f'Initial {candidate} was fragile under {name}; retain baseline.'}
sensitivity_summary = pd.DataFrame(sensitivity_rows)
display(sensitivity_summary.round(6))
peak_sensitivities = []
for quantile in [.85, .95]:
    altered = validation_predictions.copy()
    for origin in ORIGINS:
        threshold = train['aggregate_cpu_utilization'].iloc[:origin].quantile(quantile)
        mask = altered['origin'].eq(origin) & altered['target'].eq('aggregate_cpu_utilization')
        altered.loc[mask, 'is_peak'] = altered.loc[mask, 'actual'].ge(threshold)
    table = summarize(altered)
    table['peak_quantile'] = quantile
    peak_sensitivities.append(table.loc[table['target'].eq('aggregate_cpu_utilization')])
peak_sensitivity_summary = pd.concat(peak_sensitivities, ignore_index=True)
display(peak_sensitivity_summary[['candidate', 'peak_quantile', 'peak_n', 'peak_mae', 'peak_underprediction']].round(6))
sensitivity_summary.to_csv(OUTPUT_DIR / 'sensitivity_summary.csv', index=False)
peak_sensitivity_summary.to_csv(OUTPUT_DIR / 'peak_sensitivity_summary.csv', index=False)
print('Final development decisions:', json.dumps(final_decisions, indent=2))


## 14. One operational policy with declared costs

Alert for operator review if any predicted cohort mean in the next two hours reaches the
available training-prefix 90th percentile. This means **unusually high selected-cohort
utilization**, not confirmed host saturation or an SLA breach. No automatic core purchase
or scaling action follows from bucketed capacity metadata.

Score one event per validation window: did any actual mean cross that same threshold?
Assume a missed event costs 10 units and an unnecessary alert costs 1. Compare selected
forecast, persistence, always-alert and never-alert; repeat with miss cost 3. These are
illustrative alert costs, not measured infrastructure costs. Scarce event/non-event windows
cannot demonstrate robust operational value. Validation policy results also reuse the
model-selection windows and must be described as development evidence.


In [ ]:
chosen_cpu = final_decisions['aggregate_cpu_utilization']['candidate']
def alert_outcome(event, alert):
    return {'miss': int(event and not alert), 'false_alert': int(alert and not event)}

policy_events = []
for origin in ORIGINS:
    group = validation_predictions.loc[
        validation_predictions['target'].eq('aggregate_cpu_utilization') &
        validation_predictions['candidate'].eq(chosen_cpu) & validation_predictions['origin'].eq(origin)]
    threshold = float(group['peak_threshold'].iloc[0])
    event = bool(group['actual'].max() >= threshold)
    alerts = {'selected_forecast': bool(group['prediction'].max() >= threshold),
              'persistence': bool(group['baseline'].max() >= threshold),
              'always_alert': True, 'never_alert': False}
    for policy, alert in alerts.items():
        policy_events.append({'origin': origin, 'policy': policy, 'threshold': threshold,
                              'event': event, 'alert': alert, **alert_outcome(event, alert)})
policy_events = pd.DataFrame(policy_events)
policy_summary = policy_events.groupby('policy').agg(
    windows=('event', 'size'), event_windows=('event', 'sum'), alerts=('alert', 'sum'),
    misses=('miss', 'sum'), false_alerts=('false_alert', 'sum')).reset_index()
for miss_cost in [3, 10]:
    policy_summary[f'cost_miss_{miss_cost}_false_1'] = miss_cost * policy_summary['misses'] + policy_summary['false_alerts']
display(policy_events)
display(policy_summary)
policy_summary.to_csv(OUTPUT_DIR / 'policy_summary.csv', index=False)
event_count = int(policy_events.loc[policy_events['policy'].eq('persistence'), 'event'].sum())
print(f'Observed event windows: {event_count}/{len(ORIGINS)}. Three windows are insufficient for a strong operational benefit claim.')


## 15. Future-value perturbation and tiny-population tests

For every origin and candidate, perturb all later synthetic target values and check the
candidate's forecast **and fitted parameters** remain identical. At the final split,
perturb only the holdout and check development scores/model choices remain identical.
Evaluation metrics are allowed to change when their actuals change.

A raw two-VM fixture uses an offset timestamp grid and counts other than 10,000. The
same aggregation and forecast functions must work. Metadata summaries cannot become
predictors: model inputs are a one-dimensional historical target and a fixed four-column
feature matrix. The scalar cost example separately checks one miss and one false alert.


In [ ]:
toy_index = np.arange(TRAIN_SIZE + HORIZON)
toy = pd.DataFrame({'timestamp': 9000 + CADENCE * toy_index,
                    'active_vm_count': np.full(len(toy_index), 2),
                    'aggregate_cpu_utilization': 20 + .02 * toy_index + np.sin(toy_index / 5)})
for origin in ORIGINS:
    changed = toy.copy()
    changed.loc[origin:, 'aggregate_cpu_utilization'] = 99.
    changed.loc[origin:, 'active_vm_count'] = 19.
    for target in TARGETS:
        for candidate in CANDIDATES:
            before, before_info = forecast_values(toy[target].iloc[:origin], candidate, target)
            after, after_info = forecast_values(changed[target].iloc[:origin], candidate, target)
            np.testing.assert_array_equal(before, after)
            assert before_info == after_info
changed = toy.copy()
changed.loc[TRAIN_SIZE:, TARGETS] = [77., 99.]
toy_train, toy_future = development_partition(toy)
changed_train, changed_future = development_partition(changed)
toy_cv, _ = backtest(toy_train)
changed_cv, _ = backtest(changed_train)
pd.testing.assert_frame_equal(toy_cv, changed_cv)
assert choose_models(toy_cv)[0] == choose_models(changed_cv)[0]
pd.testing.assert_frame_equal(toy_future, changed_future)
toy_raw = pd.DataFrame([
    [int(row.timestamp), vm_id, float(row.aggregate_cpu_utilization),
     float(row.aggregate_cpu_utilization), float(row.aggregate_cpu_utilization)]
    for row in toy.itertuples(index=False) for vm_id in ['tiny_a', 'tiny_b']
], columns=CPU_COLUMNS)
tiny_targets, _, _ = aggregate_targets(toy_raw)
tiny_train, tiny_future = development_partition(tiny_targets)
assert tiny_train['active_vm_count'].eq(2).all()
assert tiny_future['timestamp'].iloc[0] == 9000 + TRAIN_SIZE * CADENCE
for candidate in CANDIDATES:
    tiny_forecast, _ = forecast_values(tiny_train['active_vm_count'], candidate, 'active_vm_count')
    np.testing.assert_array_equal(tiny_forecast, np.full(HORIZON, 2.))
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temporary:
    directory = Path(temporary)
    toy_meta = pd.DataFrame([
        {'vm_id': vm_id, 'subscription_id': 'sub', 'deployment_id': 'dep',
         'timestamp_vm_created': 0, 'timestamp_vm_deleted': 999999,
         'max_cpu': 50., 'avg_cpu': 20., 'p95_max_cpu': 40.,
         'vm_category': 'Unknown', 'virtual_core_bucket': '2', 'memory_gb_bucket': '4'}
        for vm_id in ['tiny_a', 'tiny_b']
    ])
    toy_raw.to_csv(directory / 'vm_cpu_readings.csv.gz', index=False)
    toy_meta.to_csv(directory / 'vm_metadata.csv.gz', index=False)
    before_train, before_template, _, _, _ = load_development(directory)
    altered_raw = toy_raw.copy()
    mask = altered_raw['timestamp'] >= int(tiny_future['timestamp'].iloc[0])
    altered_raw.loc[mask, ['min_cpu', 'max_cpu', 'avg_cpu']] = [0., 100., 99.]
    altered_raw.to_csv(directory / 'vm_cpu_readings.csv.gz', index=False)
    toy_meta[['max_cpu', 'avg_cpu', 'p95_max_cpu']] = 99.
    toy_meta['timestamp_vm_deleted'] = 99999999
    toy_meta.to_csv(directory / 'vm_metadata.csv.gz', index=False)
    after_train, after_template, _, _, _ = load_development(directory)
    pd.testing.assert_frame_equal(before_train, after_train)
    pd.testing.assert_frame_equal(before_template, after_template)
    before_cv, _ = backtest(before_train)
    after_cv, _ = backtest(after_train)
    pd.testing.assert_frame_equal(before_cv, after_cv)
    assert choose_models(before_cv)[0] == choose_models(after_cv)[0]
    for target in TARGETS:
        for candidate in CANDIDATES:
            np.testing.assert_array_equal(
                forecast_values(before_train[target], candidate, target)[0],
                forecast_values(after_train[target], candidate, target)[0])
outcomes = [alert_outcome(True, False), alert_outcome(False, True)]
assert outcomes == [{'miss': 1, 'false_alert': 0}, {'miss': 0, 'false_alert': 1}]
assert sum(10 * row['miss'] + row['false_alert'] for row in outcomes) == 11
print('Future-value, fitted-parameter, zero-baseline, feature-alignment, duplicate, gap and tiny-population checks passed.')


## 16. Freeze the development decision and produce `predictions.csv`

Save the notebook before running this cell so its source digest records the code you intend
to reproduce. Refit the selected model on all 110 development points, forecast 24 steps,
check the exact output contract, and write the file **without an index**. Predictions and
configuration are hashed before any optional final evaluation.

The run records dataset/source hashes, package versions, selection rules, chosen models and
fit diagnostics. A previous holdout-opening marker is preserved and explicitly reported;
it is not erased on rerun. If you change code after seeing the holdout, report that exposure
and do not present the new test score as a fresh untouched evaluation.


In [ ]:
def check_predictions(predictions, expected_timestamps):
    expected = np.asarray(expected_timestamps, dtype='int64')
    if list(predictions.columns) != ['timestamp', *TARGETS] or len(predictions) != len(expected):
        raise ValueError('Wrong output columns or number of rows.')
    if predictions['timestamp'].duplicated().any() or not predictions['timestamp'].is_monotonic_increasing:
        raise ValueError('Prediction timestamps must be unique and increasing.')
    if not np.array_equal(predictions['timestamp'].to_numpy(), expected):
        raise ValueError('Prediction timestamps do not match the holdout template.')
    if not np.isfinite(predictions.to_numpy(dtype=float)).all():
        raise ValueError('All output values must be finite.')
    if not predictions['active_vm_count'].ge(0).all() or not predictions['aggregate_cpu_utilization'].between(0, 100).all():
        raise ValueError('Predictions violate target bounds.')

predictions = future_timestamps.copy()
final_fits = {}
for target in TARGETS:
    predictions[target], final_fits[target] = forecast_values(train[target], final_decisions[target]['candidate'], target)
assert len(predictions) == HORIZON
check_predictions(predictions, future_timestamps['timestamp'])
prediction_path = OUTPUT_DIR / 'predictions.csv'
predictions.to_csv(prediction_path, index=False, float_format='%.12g')
check_predictions(pd.read_csv(prediction_path), future_timestamps['timestamp'])

notebook_path = PROJECT_ROOT / 'notebooks/01_forecasting.ipynb'
def source_digest(path):
    if path.suffix == '.ipynb':
        notebook = json.loads(path.read_text(encoding='utf-8'))
        development_code = [''.join(cell['source']) if isinstance(cell['source'], list) else cell['source'] for cell in notebook['cells']
                            if cell['cell_type'] == 'code' and 'holdout-evaluation' not in cell.get('metadata', {}).get('tags', [])]
        return hashlib.sha256(json.dumps(development_code, sort_keys=True).encode()).hexdigest()
    return sha256_file(path)

source_files = [notebook_path, PROJECT_ROOT / 'requirements.txt', PROJECT_ROOT / 'run_notebook.py']
source_hashes = {str(path.relative_to(PROJECT_ROOT)): source_digest(path) for path in source_files if path.exists()}
git_commit = None
if shutil.which('git'):
    git = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=False)
    git_commit = git.stdout.strip() if git.returncode == 0 else None
manifest = {
    'configuration': CONFIG, 'decisions': final_decisions, 'data_sha256': data_hashes,
    'source_sha256': source_hashes, 'git_commit': git_commit,
    'python': platform.python_version(), 'platform': platform.platform(),
    'versions': {name: metadata.version(name) for name in EXPECTED_VERSIONS},
    'predictions_sha256': sha256_file(prediction_path), 'final_fit_diagnostics': final_fits,
    'holdout_previously_opened': (OUTPUT_DIR / 'holdout_opened.json').exists(),
    'final_alert_threshold': float(train['aggregate_cpu_utilization'].quantile(PEAK_QUANTILE)),
    'final_alert': bool(predictions['aggregate_cpu_utilization'].max() >= train['aggregate_cpu_utilization'].quantile(PEAK_QUANTILE)),
}
(OUTPUT_DIR / 'frozen_run.json').write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(OUTPUT_DIR / 'environment.lock.txt').write_text(freeze.stdout, encoding='utf-8')
bad_output = predictions.copy()
bad_output.loc[0, 'aggregate_cpu_utilization'] = np.nan
expect_value_error(lambda: check_predictions(bad_output, future_timestamps['timestamp']))
print('Frozen predictions:', prediction_path)
print('SHA-256:', manifest['predictions_sha256'])
print('High-utilization review alert:', manifest['final_alert'])
if manifest['holdout_previously_opened']:
    print('NOTICE: This output directory already contains a holdout-opening record. Do not claim a fresh untouched test.')
display(predictions.head())


## 17. Evidence handoff and report writing

Generated CSVs contain the development evidence, not a final submission narrative. Write
`reports/final_report.md` (target 4–6 pages) from the actual outputs: executive conclusion,
target semantics and curation, training workload, preprocessing/availability, methods,
validation, results, failures, operational scenario and limitations. State whether the
model clears the gate; retaining persistence is a valid scientific result.

Keep separate: **verified facts** (hashes, keys, measured metrics), **hypotheses** (why changes
occurred or why smoothing might help) and **limitations** (selection, short trace, few windows,
unknown availability of source metadata). Do not turn descriptive category differences
into causal workload claims. An alert cost calculation is not demonstrated savings.

From a clean checkout, the reproduction command is:

```bash
python run_notebook.py --data challenge_data --output reports/generated
```

This runs all checks and creates predictions while leaving holdout evaluation disabled.
For the required root-level submission file, copy the generated `predictions.csv` to the
repository root without editing its contents. The data and generated output directory
belong in `.gitignore`; retain small report figures/results intentionally when submitting.


In [ ]:
evidence = pd.DataFrame([
    ['Integrity / population', 'frozen_run.json', 'SHA-256, metadata join and key audit'],
    ['Candidate comparison', 'validation_model_summary.csv', 'MAE/RMSE/relative/peak evidence'],
    ['Window stability', 'validation_window_summary.csv', 'Only three disjoint windows'],
    ['Horizon errors', 'validation_horizon_summary.csv', 'Only three examples per horizon'],
    ['Sensitivity', 'sensitivity_summary.csv', 'Origin, latency and leave-one-window checks'],
    ['Operational scenario', 'policy_summary.csv', 'Assumed costs, sparse event evidence'],
    ['Submission', 'predictions.csv', '24 finite ordered predictions; no observed holdout labels'],
], columns=['question', 'artifact', 'interpretation'])
display(evidence)
print('No holdout target values have been used for fitting, tuning, feature selection or model selection.')


## 18. Optional final holdout evaluation — disabled by default

**Stop here while developing.** Once the notebook/configuration and prediction file are
frozen, set `EVALUATE_FROZEN_HOLDOUT = True` in the next cell and run that cell only.
The headless equivalent is `python run_notebook.py --evaluate-holdout` after freezing.

This checks the frozen prediction hash before loading the holdout target table. It saves
the first opening record without overwriting it and evaluates the already generated
predictions against persistence. It does not refit or select a model. A final test is one
two-hour window, and an alert outcome there is one event, not strong deployment evidence.
If results disappoint, report them; do not tune against this test window.


In [ ]:
EVALUATE_FROZEN_HOLDOUT = os.environ.get('CWFC_EVALUATE_HOLDOUT', '0') == '1'
if EVALUATE_FROZEN_HOLDOUT:
    frozen = json.loads((OUTPUT_DIR / 'frozen_run.json').read_text(encoding='utf-8'))
    if sha256_file(prediction_path) != frozen['predictions_sha256']:
        raise ValueError('Prediction bytes changed after freezing.')
    if ensure_challenge_release(DATA_DIR) != frozen['data_sha256']:
        raise ValueError('Dataset changed after freezing.')
    for name, digest in frozen['source_sha256'].items():
        if source_digest(PROJECT_ROOT / name) != digest:
            raise ValueError(f'Source changed after freezing: {name}. Preserve the original run; do not present a retuned result as untouched.')
    opening = OUTPUT_DIR / 'holdout_opened.json'
    if not opening.exists():
        opening.write_text(json.dumps({'predictions_sha256': frozen['predictions_sha256'],
                                       'source_sha256': frozen['source_sha256'],
                                       'configuration': frozen['configuration']}, indent=2), encoding='utf-8')
    raw_for_evaluation = pd.read_csv(DATA_DIR / 'vm_cpu_readings.csv.gz', dtype={'vm_id': 'category'})
    full_for_evaluation, _, _ = aggregate_targets(raw_for_evaluation)
    holdout = full_for_evaluation.iloc[TRAIN_SIZE:].reset_index(drop=True)
    np.testing.assert_array_equal(holdout['timestamp'], predictions['timestamp'])
    holdout_rows = []
    holdout_horizons = []
    for target in TARGETS:
        baseline, _ = forecast_values(train[target], 'persistence', target)
        metrics = error_metrics(holdout[target], predictions[target], baseline)
        threshold = float(train[target].quantile(PEAK_QUANTILE))
        peaks = holdout[target].ge(threshold).to_numpy() if target == 'aggregate_cpu_utilization' else np.zeros(HORIZON, dtype=bool)
        holdout_rows.append({'target': target, 'candidate': frozen['decisions'][target]['candidate'], **metrics,
                             'peak_n': int(peaks.sum()),
                             'peak_mae': float(np.mean(np.abs(holdout[target].to_numpy()[peaks] - predictions[target].to_numpy()[peaks]))) if peaks.any() else np.nan,
                             'peak_rmse': float(np.sqrt(np.mean((holdout[target].to_numpy()[peaks] - predictions[target].to_numpy()[peaks])**2))) if peaks.any() else np.nan,
                             'peak_underprediction': float(np.mean(np.maximum(holdout[target].to_numpy()[peaks] - predictions[target].to_numpy()[peaks], 0))) if peaks.any() else np.nan})
        for h, (observed, predicted, base) in enumerate(zip(holdout[target], predictions[target], baseline), 1):
            holdout_horizons.append({'target': target, 'horizon': h, 'actual': observed,
                                     'prediction': predicted, 'baseline': base,
                                     'absolute_error': abs(predicted - observed)})
    holdout_summary = pd.DataFrame(holdout_rows)
    holdout_summary.to_csv(OUTPUT_DIR / 'holdout_metrics.csv', index=False)
    pd.DataFrame(holdout_horizons).to_csv(OUTPUT_DIR / 'holdout_horizon_errors.csv', index=False)
    display(holdout_summary)
    observed_event = bool(holdout['aggregate_cpu_utilization'].max() >= frozen['final_alert_threshold'])
    baseline_alert = bool(train['aggregate_cpu_utilization'].iloc[-1] >= frozen['final_alert_threshold'])
    holdout_policies = []
    for policy, alert in {'selected_forecast': frozen['final_alert'], 'persistence': baseline_alert,
                          'always_alert': True, 'never_alert': False}.items():
        result = {'policy': policy, 'event': observed_event, 'alert': alert, **alert_outcome(observed_event, alert)}
        result['cost_miss_10_false_1'] = 10 * result['miss'] + result['false_alert']
        holdout_policies.append(result)
    holdout_policy_summary = pd.DataFrame(holdout_policies)
    holdout_policy_summary.to_csv(OUTPUT_DIR / 'holdout_policy.csv', index=False)
    display(holdout_policy_summary)
    print('Final test evaluated. Do not use these results to revise models, features or thresholds.')
    del raw_for_evaluation, full_for_evaluation
else:
    print('Holdout remains unopened for evaluation. Development and predictions are complete.')
